# Tarea 1: Pipeline ETL y Estructuración del Repositorio

Este notebook contiene el proceso completo de extracción, transformación y carga (ETL) para la Plataforma Predictiva del Mundial de Fútbol 2026. A partir de los datos históricos de partidos y calificaciones de ELO, construimos un dataset consolidado libre de *data leakage*.

In [ ]:
import os
import numpy as np
import pandas as pd

# Rutas principales
RAW_DIR = "../data/raw"
PROCESSED_DIR = "../data/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

results_path = os.path.join(RAW_DIR, "results.csv")
former_names_path = os.path.join(RAW_DIR, "former_names.csv")
shootouts_path = os.path.join(RAW_DIR, "shootouts.csv")

### 1. Carga de Datos y Limpieza Básica

Cargamos los datasets originales, formateamos las fechas, removemos duplicados y filtramos filas con marcadores nulos.

In [ ]:
print("Cargando datasets...")
df = pd.read_csv(results_path)
former = pd.read_csv(former_names_path)
shootouts = pd.read_csv(shootouts_path)

print(f"Partidos cargados inicialmente: {len(df)}")

# Formato de fechas
df['date'] = pd.to_datetime(df['date'], format='mixed')
former['start_date'] = pd.to_datetime(former['start_date'])
former['end_date'] = pd.to_datetime(former['end_date'])
shootouts['date'] = pd.to_datetime(shootouts['date'])

# Eliminar nulos en goles
df = df.dropna(subset=['home_score', 'away_score'])

# Limpiar espacios vacíos en columnas categóricas
for col in ['home_team', 'away_team', 'tournament', 'city', 'country']:
    df[col] = df[col].astype(str).str.strip()

# Eliminar duplicados
df = df.drop_duplicates()
print(f"Partidos tras limpieza inicial: {len(df)}")

### 2. Unificación de Nombres de Selecciones

Usamos el archivo de nombres históricos (`former_names.csv`) para mapear nombres antiguos de selecciones a sus nombres vigentes según la fecha del partido.

In [ ]:
print("Unificando nombres de países según rango de fechas...")
for idx, row in former.iterrows():
    mask_home = (df['home_team'] == row['former']) & (df['date'] >= row['start_date']) & (df['date'] <= row['end_date'])
    df.loc[mask_home, 'home_team'] = row['current']
    
    mask_away = (df['away_team'] == row['former']) & (df['date'] >= row['start_date']) & (df['date'] <= row['end_date'])
    df.loc[mask_away, 'away_team'] = row['current']
print("Unificación de nombres completada.")

### 3. Creación de la Variable Objetivo (`result`)

Definimos el resultado como: 
- `0` = derrota local (gana visitante)
- `1` = empate
- `2` = victoria local

In [ ]:
def determinar_resultado(row):
    if row['home_score'] > row['away_score']:
        return 2
    elif row['home_score'] == row['away_score']:
        return 1
    else:
        return 0

df['result'] = df.apply(determinar_resultado, axis=1)

# Ordenar cronológicamente
df = df.sort_values('date').reset_index(drop=True)
print("Variable objetivo 'result' generada y datos ordenados.")

### 4. Cálculo de Sistema ELO con K Diferenciado

Calculamos los ratings ELO de cada selección dinámicamente antes del inicio del partido, asegurando que no haya filtración de datos (*data leakage*).

In [ ]:
print("Inicializando y ejecutando simulación ELO...")
K_TOURNAMENT = {
    'FIFA World Cup': 60,
    'UEFA Euro': 40,
    'Copa América': 40,
    'Friendly': 20,
    'default': 30
}

elo_ratings = {}
elo_home_list = []
elo_away_list = []

for idx, row in df.iterrows():
    home = row['home_team']
    away = row['away_team']
    tourn = row['tournament']
    
    if home not in elo_ratings:
        elo_ratings[home] = 1500.0
    if away not in elo_ratings:
        elo_ratings[away] = 1500.0
        
    elo_h = elo_ratings[home]
    elo_a = elo_ratings[away]
    
    elo_home_list.append(elo_h)
    elo_away_list.append(elo_a)
    
    # Fórmulas ELO estándar
    E_home = 1.0 / (1.0 + 10.0 ** ((elo_a - elo_h) / 400.0))
    E_away = 1.0 / (1.0 + 10.0 ** ((elo_h - elo_a) / 400.0))
    
    res = row['result']
    S_home = 1.0 if res == 2 else (0.5 if res == 1 else 0.0)
    S_away = 1.0 - S_home
    
    K = K_TOURNAMENT.get(tourn, K_TOURNAMENT['default'])
    
    elo_ratings[home] = elo_h + K * (S_home - E_home)
    elo_ratings[away] = elo_a + K * (S_away - E_away)

df['elo_home'] = elo_home_list
df['elo_away'] = elo_away_list
df['elo_diff'] = df['elo_home'] - df['elo_away']
print("Cálculo de ELO completado.")

### 5. Cálculo de Forma Reciente

Definimos la forma reciente de cada equipo usando ventanas móviles de sus últimos 5, 10 y 20 partidos históricos jugados antes de la fecha del encuentro actual.

In [ ]:
print("Calculando variables de forma reciente...")
team_history = {}
features_recent = {
    'home_wins_5': [], 'home_draws_5': [], 'home_losses_5': [],
    'home_goals_scored_5': [], 'home_goals_conceded_5': [],
    'home_wins_10': [], 'home_wins_20': [],
    'away_wins_5': [], 'away_draws_5': [], 'away_losses_5': [],
    'away_goals_scored_5': [], 'away_goals_conceded_5': [],
    'away_wins_10': [], 'away_wins_20': []
}

for idx, row in df.iterrows():
    home = row['home_team']
    away = row['away_team']
    res = row['result']
    home_g = row['home_score']
    away_g = row['away_score']
    
    hist_home = team_history.get(home, [])
    hist_away = team_history.get(away, [])
    
    # Home windows
    w5_h = hist_home[-5:] if len(hist_home) >= 5 else hist_home
    if len(w5_h) > 0:
        features_recent['home_wins_5'].append(sum(1 for x in w5_h if x[0] == 2))
        features_recent['home_draws_5'].append(sum(1 for x in w5_h if x[0] == 1))
        features_recent['home_losses_5'].append(sum(1 for x in w5_h if x[0] == 0))
        features_recent['home_goals_scored_5'].append(sum(x[1] for x in w5_h))
        features_recent['home_goals_conceded_5'].append(sum(x[2] for x in w5_h))
    else:
        for key in ['home_wins_5', 'home_draws_5', 'home_losses_5', 'home_goals_scored_5', 'home_goals_conceded_5']:
            features_recent[key].append(0)
            
    w10_h = hist_home[-10:] if len(hist_home) >= 10 else hist_home
    features_recent['home_wins_10'].append(sum(1 for x in w10_h if x[0] == 2) if len(w10_h) > 0 else 0)
    w20_h = hist_home[-20:] if len(hist_home) >= 20 else hist_home
    features_recent['home_wins_20'].append(sum(1 for x in w20_h if x[0] == 2) if len(w20_h) > 0 else 0)
    
    # Away windows
    w5_a = hist_away[-5:] if len(hist_away) >= 5 else hist_away
    if len(w5_a) > 0:
        features_recent['away_wins_5'].append(sum(1 for x in w5_a if x[0] == 2))
        features_recent['away_draws_5'].append(sum(1 for x in w5_a if x[0] == 1))
        features_recent['away_losses_5'].append(sum(1 for x in w5_a if x[0] == 0))
        features_recent['away_goals_scored_5'].append(sum(x[1] for x in w5_a))
        features_recent['away_goals_conceded_5'].append(sum(x[2] for x in w5_a))
    else:
        for key in ['away_wins_5', 'away_draws_5', 'away_losses_5', 'away_goals_scored_5', 'away_goals_conceded_5']:
            features_recent[key].append(0)
            
    w10_a = hist_away[-10:] if len(hist_away) >= 10 else hist_away
    features_recent['away_wins_10'].append(sum(1 for x in w10_a if x[0] == 2) if len(w10_a) > 0 else 0)
    w20_a = hist_away[-20:] if len(hist_away) >= 20 else hist_away
    features_recent['away_wins_20'].append(sum(1 for x in w20_a if x[0] == 2) if len(w20_a) > 0 else 0)
    
    # Guardar en históricos
    if home not in team_history: team_history[home] = []
    team_history[home].append((res, home_g, away_g))
    if away not in team_history: team_history[away] = []
    team_history[away].append((2 - res, away_g, home_g))

for k, v in features_recent.items():
    df[k] = v
print("Variables de racha generadas.")

### 6. Historial de Enfrentamientos Directos (H2H)

Calculamos el rendimiento en los últimos 10 enfrentamientos directos cara a cara entre ambas selecciones.

In [ ]:
print("Calculando métricas H2H...")
h2h_history = {}
h2h_home_wins, h2h_draws, h2h_away_wins = [], [], []
h2h_home_goals_avg, h2h_away_goals_avg = [], []

for idx, row in df.iterrows():
    home = row['home_team']
    away = row['away_team']
    home_g = row['home_score']
    away_g = row['away_score']
    
    pair = frozenset([home, away])
    past_h2h = h2h_history.get(pair, [])
    
    last_10 = past_h2h[-10:] if len(past_h2h) >= 10 else past_h2h
    
    if len(last_10) > 0:
        h_w, d_c, a_w = 0, 0, 0
        h_goals, a_goals = 0, 0
        for _, p_home, p_away, p_h_score, p_a_score in last_10:
            if p_home == home:
                h_goals += p_h_score
                a_goals += p_a_score
                if p_h_score > p_a_score: h_w += 1
                elif p_h_score == p_a_score: d_c += 1
                else: a_w += 1
            else:
                h_goals += p_a_score
                a_goals += p_h_score
                if p_a_score > p_h_score: h_w += 1
                elif p_a_score == p_h_score: d_c += 1
                else: a_w += 1
        h2h_home_wins.append(h_w)
        h2h_draws.append(d_c)
        h2h_away_wins.append(a_w)
        h2h_home_goals_avg.append(h_goals / len(last_10))
        h2h_away_goals_avg.append(a_goals / len(last_10))
    else:
        h2h_home_wins.append(0)
        h2h_draws.append(0)
        h2h_away_wins.append(0)
        h2h_home_goals_avg.append(0.0)
        h2h_away_goals_avg.append(0.0)
        
    if pair not in h2h_history: h2h_history[pair] = []
    h2h_history[pair].append((row['date'], home, away, home_g, away_g))

df['h2h_home_wins'] = h2h_home_wins
df['h2h_draws'] = h2h_draws
df['h2h_away_wins'] = h2h_away_wins
df['h2h_home_goals_avg'] = h2h_home_goals_avg
df['h2h_away_goals_avg'] = h2h_away_goals_avg
print("Variables H2H generadas.")

### 7. Variables de Contexto

Extraemos factores contextuales del encuentro: localía neutra, peso específico del torneo y tipo de fase (grupo o eliminatoria directa).

In [ ]:
print("Generando variables contextuales...")
df['is_neutral'] = df['neutral'].astype(int)

def get_tourn_weight(tourn):
    if tourn == 'FIFA World Cup': return 3
    elif tourn == 'Friendly': return 1
    else: return 5
df['tournament_weight'] = df['tournament'].apply(get_tourn_weight)

# Clasificación inteligente de Fase de Eliminación Directa (Knockout)
shootout_pairs = set()
for _, row in shootouts.iterrows():
    shootout_pairs.add((row['date'], frozenset([row['home_team'], row['away_team']])))

major_tournaments = ['FIFA World Cup', 'UEFA Euro', 'Copa América']
major_df = df[df['tournament'].isin(major_tournaments)].copy()
major_df['year'] = major_df['date'].dt.year
tournament_dates = major_df.groupby(['tournament', 'year'])['date'].agg(['min', 'max']).reset_index()

tourn_date_dict = {}
for _, row in tournament_dates.iterrows():
    tourn_date_dict[(row['tournament'], row['year'])] = (row['min'], row['max'])

def is_knockout(row):
    if (row['date'], frozenset([row['home_team'], row['away_team']])) in shootout_pairs:
        return 1
    tourn = row['tournament']
    if tourn in major_tournaments:
        yr = row['date'].year
        if (tourn, yr) in tourn_date_dict:
            t_min, t_max = tourn_date_dict[(tourn, yr)]
            total_days = (t_max - t_min).days
            if total_days > 2:
                elapsed_days = (row['date'] - t_min).days
                if elapsed_days / total_days >= 0.6:
                    return 1
    return 0

df['phase_encoded'] = df.apply(is_knockout, axis=1)
print("Variables contextuales completadas.")

### 8. Guardado del Dataset Limpio

Almacenamos el dataset procesado final en `data/processed/matches_clean.csv`.

In [ ]:
output_path = os.path.join(PROCESSED_DIR, "matches_clean.csv")
df.to_csv(output_path, index=False)
print(f"¡Éxito! Dataset limpio guardado en: {output_path} con dimensiones: {df.shape}")
print("Primeras filas del dataset procesado:")
print(df.head(3))